# Tahap 1 — Forecast demand air per area dengan XGBoost (Google Colab)
Data: `water_outage_dummy_db.xlsx` (sintetis, dikalibrasi ke konsumsi 2022 per ilçe Istanbul).

Jalankan sel dari atas ke bawah (**Runtime → Run all**). GPU tidak diperlukan; runtime CPU biasa cukup.

## Langkah 0 — Setup: install paket

In [ ]:
# xgboost, pandas, numpy, scikit-learn, scipy, matplotlib sudah ada di Colab.
# optuna & shap dipasang (atau di-upgrade) di sini.
!pip install -q optuna shap xgboost>=2.0

import json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.width", 160)
SEED = 42
print("xgboost", xgb.__version__, "| optuna", optuna.__version__)

## Langkah 1 — Load data
Pilih **salah satu**: (A) upload file langsung, atau (B) dari Google Drive.
File xlsx 12 MB dengan 341 ribu baris butuh ±1–2 menit untuk dibaca.

In [ ]:
USE_DRIVE = False   # ganti True kalau file ada di Google Drive
DRIVE_PATH = "/content/drive/MyDrive/water_outage_dummy_db.xlsx"   # sesuaikan

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    XLSX = DRIVE_PATH
else:
    XLSX = "/content/water_outage_dummy_db.xlsx"
    if not os.path.exists(XLSX):
        from google.colab import files
        up = files.upload()               # pilih water_outage_dummy_db.xlsx
        XLSX = "/content/" + list(up.keys())[0]
print("File:", XLSX)

In [ ]:
t0 = time.time()
sheets = pd.read_excel(XLSX, sheet_name=["areas", "tanks", "area_tank_links",
                                         "hourly_usage_2022", "outage_scenarios",
                                         "scenario_assignments"])
areas  = sheets["areas"]
tanks  = sheets["tanks"]
links  = sheets["area_tank_links"]
hourly = sheets["hourly_usage_2022"]
scen   = sheets["outage_scenarios"]
asgn   = sheets["scenario_assignments"]
hourly["timestamp"] = pd.to_datetime(hourly["timestamp"])
scen["outage_start"] = pd.to_datetime(scen["outage_start"])
print(f"Loaded in {time.time()-t0:.0f}s")
for n, d in sheets.items():
    print(f"{n:22s} {d.shape}")
hourly.head()

## Langkah 2 — Join hourly dengan areas

In [ ]:
STATIC = ["population_2022", "households_est", "lpcd_2022", "share_residential",
          "share_commercial", "share_industrial", "hospitals", "schools", "priority_weight"]
df = hourly.merge(areas[["area_id"] + STATIC], on="area_id", how="left")
df["area_id"] = df["area_id"].astype("category")
df = df.sort_values(["area_id", "timestamp"]).reset_index(drop=True)
assert df[STATIC].isna().sum().sum() == 0
print(df.shape)

## Langkah 3 — Fitur waktu

In [ ]:
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
df["sin_hour"] = np.sin(2*np.pi*df["hour"]/24)
df["cos_hour"] = np.cos(2*np.pi*df["hour"]/24)
df["sin_dow"]  = np.sin(2*np.pi*df["day_of_week"]/7)
df["cos_dow"]  = np.cos(2*np.pi*df["day_of_week"]/7)
df["temp_above18"] = (df["temperature_c"] - 18).clip(lower=0)

## Langkah 4 — Fitur lag (anti-bocor)
Pemadaman berlangsung maksimal 24 jam, jadi semua lag **≥ 24 jam**. Dengan begitu setiap fitur sudah diketahui saat pemadaman dimulai.
Rolling mean dihitung dari deret yang sudah di-shift 24 jam.

In [ ]:
g = df.groupby("area_id", observed=True)["usage_m3"]
df["lag_24"]  = g.shift(24)
df["lag_48"]  = g.shift(48)
df["lag_168"] = g.shift(168)
s24 = g.shift(24)
df["roll_mean_24"]  = s24.groupby(df["area_id"], observed=True).transform(lambda s: s.rolling(24).mean())
df["roll_mean_168"] = s24.groupby(df["area_id"], observed=True).transform(lambda s: s.rolling(168).mean())
df["roll_std_24"]   = s24.groupby(df["area_id"], observed=True).transform(lambda s: s.rolling(24).std())

# minggu pertama tidak punya lag lengkap -> dibuang
df = df.dropna(subset=["lag_168", "roll_mean_168"]).reset_index(drop=True)
print(df.shape, df["timestamp"].min())

## Langkah 5 — Target dinormalisasi per area
`y = usage_m3 / rata-rata area (dihitung dari train saja)`. Fitur lag juga dibagi dengan angka yang sama, sehingga satu model global bisa belajar pola dari 39 area dengan skala berbeda.

In [ ]:
SPLIT_DATE = pd.Timestamp("2022-11-01")
area_mean = df[df["timestamp"] < SPLIT_DATE].groupby("area_id", observed=True)["usage_m3"].mean()
df["area_mean"] = df["area_id"].map(area_mean).astype(float)

LAGS = ["lag_24", "lag_48", "lag_168", "roll_mean_24", "roll_mean_168", "roll_std_24"]
for c in LAGS:
    df[c + "_n"] = df[c] / df["area_mean"]
df["y"] = df["usage_m3"] / df["area_mean"]

FEATURES = (["area_id", "hour", "day_of_week", "month", "is_holiday", "is_weekend",
             "sin_hour", "cos_hour", "sin_dow", "cos_dow", "temperature_c", "temp_above18"]
            + STATIC + [c + "_n" for c in LAGS])
print(len(FEATURES), "fitur")

## Langkah 6 — Split berdasarkan waktu (jangan diacak)

In [ ]:
train = df[df["timestamp"] <  SPLIT_DATE].reset_index(drop=True)
test  = df[df["timestamp"] >= SPLIT_DATE].reset_index(drop=True)
X_tr, y_tr = train[FEATURES], train["y"]
X_te, y_te = test[FEATURES],  test["y"]
print("train", X_tr.shape, train.timestamp.min(), "->", train.timestamp.max())
print("test ", X_te.shape, test.timestamp.min(),  "->", test.timestamp.max())

## Langkah 7 — Metrik & validasi expanding window (TimeSeriesSplit)

In [ ]:
def wape(actual, pred):
    actual, pred = np.asarray(actual), np.asarray(pred)
    return np.abs(actual - pred).sum() / np.abs(actual).sum()

# Split berdasarkan jam unik (bukan baris) agar semua area di jam yang sama ada di fold yang sama
uniq_ts = np.sort(train["timestamp"].unique())
tscv = TimeSeriesSplit(n_splits=5)
FOLDS = []
for tr_i, va_i in tscv.split(uniq_ts):
    tr_mask = train["timestamp"].isin(uniq_ts[tr_i]).values
    va_mask = train["timestamp"].isin(uniq_ts[va_i]).values
    FOLDS.append((np.where(tr_mask)[0], np.where(va_mask)[0]))
for k, (a, b) in enumerate(FOLDS):
    print(f"fold {k}: train {len(a):>7,} rows | valid {len(b):>6,} rows | valid mulai {train.timestamp.iloc[b[0]]}")

## Langkah 8 — Tuning hyperparameter dengan Optuna
`N_TRIALS = 30` butuh ±10–20 menit di CPU Colab. Untuk uji cepat, set ke 5.
WAPE dihitung dalam m³ (prediksi × rata-rata area), bukan pada target ternormalisasi.

In [ ]:
N_TRIALS = 30

def objective(trial):
    params = dict(
        n_estimators=2000,
        learning_rate=trial.suggest_float("learning_rate", 0.03, 0.2, log=True),
        max_depth=trial.suggest_int("max_depth", 4, 10),
        min_child_weight=trial.suggest_float("min_child_weight", 1, 20, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
        tree_method="hist", enable_categorical=True, early_stopping_rounds=50,
        random_state=SEED, n_jobs=-1,
    )
    scores, iters = [], []
    for tr_i, va_i in FOLDS[-3:]:          # 3 fold terakhir agar lebih cepat
        m = xgb.XGBRegressor(**params)
        m.fit(X_tr.iloc[tr_i], y_tr.iloc[tr_i],
              eval_set=[(X_tr.iloc[va_i], y_tr.iloc[va_i])], verbose=False)
        p = m.predict(X_tr.iloc[va_i]) * train["area_mean"].iloc[va_i].values
        scores.append(wape(train["usage_m3"].iloc[va_i], p))
        iters.append(m.best_iteration)
    trial.set_user_attr("n_estimators", int(np.mean(iters)) + 1)
    return float(np.mean(scores))

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
best_params = study.best_params
best_n = study.best_trial.user_attrs["n_estimators"]
print("CV WAPE terbaik:", round(study.best_value, 4))
print("Parameter:", best_params, "| n_estimators:", best_n)

## Langkah 9 — Latih model akhir pada seluruh train

In [ ]:
# 10% jam terakhir train dipakai sebagai eval_set untuk early stopping
cut = train["timestamp"].quantile(0.9)
fit_idx = train["timestamp"] <  cut
val_idx = train["timestamp"] >= cut

model = xgb.XGBRegressor(**best_params, n_estimators=max(best_n * 2, 300),
                         tree_method="hist", enable_categorical=True,
                         early_stopping_rounds=50, random_state=SEED, n_jobs=-1)
model.fit(X_tr[fit_idx], y_tr[fit_idx],
          eval_set=[(X_tr[fit_idx], y_tr[fit_idx]), (X_tr[val_idx], y_tr[val_idx])],
          verbose=200)
print("best_iteration:", model.best_iteration)

hist = model.evals_result()
plt.figure(figsize=(7,3))
plt.plot(hist["validation_0"]["rmse"], label="train")
plt.plot(hist["validation_1"]["rmse"], label="valid")
plt.xlabel("iterasi"); plt.ylabel("RMSE (target ternormalisasi)"); plt.legend(); plt.title("Kurva belajar")
plt.show()

## Langkah 10 — Evaluasi pada test (Nov–Des 2022)
Dibandingkan dengan baseline naif: **nilai jam yang sama minggu lalu** (`lag_168`).

In [ ]:
test["pred_m3"] = model.predict(X_te) * test["area_mean"].values
test["naive_m3"] = test["lag_168"]

res = pd.DataFrame({
    "MAE (m3/jam)": [mean_absolute_error(test.usage_m3, test.pred_m3),
                     mean_absolute_error(test.usage_m3, test.naive_m3)],
    "WAPE":         [wape(test.usage_m3, test.pred_m3), wape(test.usage_m3, test.naive_m3)],
}, index=["XGBoost", "Naif (minggu lalu)"])
display(res.style.format({"MAE (m3/jam)": "{:,.1f}", "WAPE": "{:.2%}"}))

# per area
per_area = (test.groupby("area_id", observed=True)
                .apply(lambda d: pd.Series({"WAPE_xgb": wape(d.usage_m3, d.pred_m3),
                                            "WAPE_naif": wape(d.usage_m3, d.naive_m3)}))
                .merge(areas.set_index("area_id")[["district"]], left_index=True, right_index=True)
                .sort_values("WAPE_xgb", ascending=False))
display(per_area.head(10).style.format({"WAPE_xgb": "{:.2%}", "WAPE_naif": "{:.2%}"}))

In [ ]:
# Error per jam & contoh deret satu area
fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
by_h = test.assign(ae=(test.usage_m3 - test.pred_m3).abs()).groupby("hour")[["ae", "usage_m3"]].sum()
ax[0].bar(by_h.index, by_h.ae / by_h.usage_m3)
ax[0].set_title("WAPE per jam"); ax[0].set_xlabel("jam")

a0 = per_area.index[0]
d = test[(test.area_id == a0) & (test.timestamp < SPLIT_DATE + pd.Timedelta(days=7))]
ax[1].plot(d.timestamp, d.usage_m3, label="aktual")
ax[1].plot(d.timestamp, d.pred_m3, label="XGBoost")
ax[1].plot(d.timestamp, d.naive_m3, label="naif", alpha=.5)
ax[1].set_title(f"{a0} – minggu pertama test"); ax[1].legend()
plt.tight_layout(); plt.show()

### Evaluasi agregat per durasi pemadaman
Tahap 2 butuh **total demand selama pemadaman**, bukan per jam. Di sini dicek error total untuk jendela 4/6/8/12/24 jam.

In [ ]:
rows = []
for dur in [4, 6, 8, 12, 24]:
    t = test.copy()
    t["block"] = ((t["timestamp"] - SPLIT_DATE) / pd.Timedelta(hours=dur)).astype(int)
    agg = t.groupby(["area_id", "block"], observed=True)[["usage_m3", "pred_m3", "naive_m3"]].sum()
    rows.append({"durasi_jam": dur, "WAPE_xgb": wape(agg.usage_m3, agg.pred_m3),
                 "WAPE_naif": wape(agg.usage_m3, agg.naive_m3)})
display(pd.DataFrame(rows).style.format({"WAPE_xgb": "{:.2%}", "WAPE_naif": "{:.2%}"}))

## Langkah 11 — Interpretasi: feature importance & SHAP

In [ ]:
import shap
imp = pd.Series(model.get_booster().get_score(importance_type="gain")).sort_values()
imp.tail(15).plot.barh(figsize=(6, 5), title="Feature importance (gain)"); plt.show()

sample = X_te.sample(3000, random_state=SEED)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(sample)
shap.summary_plot(shap_values, sample, max_display=15)

## Langkah 12 — Forecast demand untuk setiap skenario pemadaman
Untuk tiap skenario: jumlahkan prediksi tiap area terdampak sepanjang `duration_h` dari `outage_start`.
Karena semua lag ≥ 24 jam, prediksi ini hanya memakai informasi yang tersedia saat pemadaman dimulai.
Catatan: skenario `train` diprediksi in-sample, jadi angka yang jujur adalah skenario `test`.

In [ ]:
df["pred_m3"] = model.predict(df[FEATURES]) * df["area_mean"].values
pred_idx = df.set_index(["area_id", "timestamp"])["pred_m3"].sort_index()
act_idx  = df.set_index(["area_id", "timestamp"])["usage_m3"].sort_index()

def window_sum(series, area, start, dur):
    return series.loc[(area, slice(start, start + pd.Timedelta(hours=dur - 1)))].sum()

meta = scen.set_index("scenario_id")[["outage_start", "duration_h", "split"]]
sa = asgn.merge(meta, left_on="scenario_id", right_index=True)
sa = sa[sa["outage_start"] >= df["timestamp"].min()].copy()   # buang skenario di minggu pertama
sa["demand_forecast_m3"] = [window_sum(pred_idx, a, s, d) for a, s, d in
                            zip(sa.area_id, sa.outage_start, sa.duration_h)]
sa["demand_check_m3"]    = [window_sum(act_idx, a, s, d) for a, s, d in
                            zip(sa.area_id, sa.outage_start, sa.duration_h)]
print("cek demand aktual cocok dengan sheet:", np.allclose(sa.demand_check_m3, sa.demand_m3, rtol=1e-3))
for sp in ["train", "test"]:
    x = sa[sa.split == sp]
    print(f"{sp:5s}: {x.scenario_id.nunique()} skenario | WAPE demand per area = {wape(x.demand_m3, x.demand_forecast_m3):.2%}")

## Langkah 13 — Uji ke Tahap 2 (MILP alokasi tangki)
Tiga kebijakan dibandingkan pada skenario test:
1. **MILP + demand aktual** (batas atas / oracle)
2. **MILP + forecast XGBoost** (kondisi realistis) — dengan buffer keamanan opsional
3. **Greedy** (dari sheet)

Setelah alokasi dibuat dengan forecast, demand **aktual** dipakai untuk menghitung kekurangan nyata: jika total demand aktual di satu tangki melebihi isinya, selisihnya dihitung sebagai *unserved*.

In [ ]:
from scipy.optimize import milp, LinearConstraint, Bounds

TANK_IDS = tanks["tank_id"].tolist()
dist = links.pivot(index="area_id", columns="tank_id", values="distance_km")[TANK_IDS]
conn = links.pivot(index="area_id", columns="tank_id", values="connected")[TANK_IDS]
prio = areas.set_index("area_id")["priority_weight"]

def solve_milp(area_ids, demand, level):
    n, m = len(area_ids), len(TANK_IDS)
    D = dist.loc[area_ids].values; C = conn.loc[area_ids].values.astype(float)
    w = prio.loc[area_ids].values
    c = (demand[:, None] * D / 100 - 1000.0 * w[:, None] * demand[:, None]).ravel() / 1e3
    A1 = np.zeros((n, n*m)); A2 = np.zeros((m, n*m))
    for i in range(n):
        A1[i, i*m:(i+1)*m] = 1
        A2[np.arange(m), i*m + np.arange(m)] = demand[i]
    r = milp(c, constraints=[LinearConstraint(A1, -np.inf, 1), LinearConstraint(A2, -np.inf, level)],
             integrality=np.ones(n*m), bounds=Bounds(0, C.ravel()), options={"time_limit": 30})
    x = np.round(r.x.reshape(n, m)).astype(int)
    return np.where(x.sum(1) > 0, x.argmax(1), -1)          # indeks tangki, -1 = tidak dilayani

def real_unserved(assign, actual, level):
    un = actual[assign < 0].sum()
    for j in range(len(level)):
        load = actual[assign == j].sum()
        un += max(0.0, load - level[j])
    return un

def evaluate(split="test", buffer=0.0):
    out = []
    for sid, g in sa[sa.split == split].groupby("scenario_id"):
        level = np.array(json.loads(scen.set_index("scenario_id").loc[sid, "tank_levels_m3"]), float)
        ids, act = g.area_id.values, g.demand_m3.values
        fc = g.demand_forecast_m3.values * (1 + buffer)
        a_oracle = solve_milp(ids, act, level)
        a_fc     = solve_milp(ids, fc, level)
        a_greedy = np.array([TANK_IDS.index(t) if t != "NONE" else -1 for t in g.greedy_tank_id])
        out.append({"scenario_id": sid, "total_demand": act.sum(),
                    "unserved_oracle": real_unserved(a_oracle, act, level),
                    "unserved_xgb":    real_unserved(a_fc, act, level),
                    "unserved_greedy": real_unserved(a_greedy, act, level)})
    return pd.DataFrame(out)

for buf in [0.0, 0.05]:
    ev = evaluate("test", buffer=buf)
    tot = ev.total_demand.sum()
    print(f"buffer {buf:.0%}: unserved / total demand  ->  oracle {ev.unserved_oracle.sum()/tot:.2%} | "
          f"MILP+XGBoost {ev.unserved_xgb.sum()/tot:.2%} | greedy {ev.unserved_greedy.sum()/tot:.2%} "
          f"({len(ev)} skenario)")

## Langkah 14 — Simpan model & hasil, lalu download

In [ ]:
model.save_model("/content/xgb_demand.json")
json.dump({"features": FEATURES, "static": STATIC, "lags": LAGS, "split_date": str(SPLIT_DATE),
           "area_mean": area_mean.to_dict(), "params": best_params},
          open("/content/xgb_demand_meta.json", "w"), indent=2, default=str)
sa.to_csv("/content/scenario_demand_forecast.csv", index=False)
test[["timestamp", "area_id", "usage_m3", "pred_m3", "naive_m3"]].to_csv("/content/test_predictions.csv", index=False)

from google.colab import files
for f in ["xgb_demand.json", "xgb_demand_meta.json", "scenario_demand_forecast.csv", "test_predictions.csv"]:
    files.download("/content/" + f)

# Cara memuat ulang nanti:
# m = xgb.XGBRegressor(); m.load_model("/content/xgb_demand.json")